# Two-Tower BPR — Modified by using weighted ratings



# 1. Notebook Overview and Introduction

**Compared to pure ratings:**
1. Used the same evaluation method: full catalog evaluation
2. Used the same model architecture, training loop, BPR loss
3. Same sementation for users: `pd.qcut` into 4 equal groups: light / medium / heavy / power with ~75 users each


**Changes from caleb's Two-Tower BPR:**
1. I used train_complete.csv. This makes `train_df` columns include the one-hot watch_status_* columns and effective_rating
2. `watching_status` weighting on ratings


# 2. Import Libraries & Mount Google Drive

In [1]:
import os
import copy
import math
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [2]:
from google.colab import drive
drive.mount('/content/drive')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

Mounted at /content/drive


# 3.Configuration and Hyperparameters

In [3]:
BASE_PATH = Path("/content/drive/MyDrive/yuran_files")   # change if needed
TRAIN_PATH = BASE_PATH / "train_complete.csv" # including all watching related info
VAL_PATH   = BASE_PATH / "val_complete.csv"
TEST_PATH  = BASE_PATH / "test_complete.csv"
MASTER_ANIME_PATH = BASE_PATH / "MASTER_ANIME_TOWER_FEATURES_1BASED.npy"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

SEED = 42
set_seed(SEED)

# data / labels
POS_THRESHOLD = 6.0   # items with score >= 6 are positive for BPR training

# training
BATCH_SIZE   = 2048
EPOCHS       = 5
LR           = 3e-4
WEIGHT_DECAY = 1e-6
PATIENCE     = 2

# model
EMB_DIM    = 64
HIDDEN_DIM = 256
DROPOUT    = 0.15

# evaluation
EVAL_K           = 10
EVAL_MAX_USERS   = 1000   # set None for all users
EVAL_NEG_SAMPLES = 1000

Using device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


#4. Load and Prepare DataFrames

In [4]:
train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

# normalize target col
for df in [train_df, val_df, test_df]:
    if "score" in df.columns and "rating" not in df.columns:
        df.rename(columns={"score": "rating"}, inplace=True)

# -------------------------------------------------------
# Also keep watching_status if present.
# If CSVs don't have it the code falls back gracefully.
# -------------------------------------------------------
one_hot_status_cols = [
    "watch_status_Completed",
    "watch_status_Currently Watching",
    "watch_status_Dropped",
    "watch_status_On Hold",
    "watch_status_Plan to Watch"
]
has_status = any(col in train_df.columns for col in one_hot_status_cols)

if has_status:
    print("One-hot watching_status columns found — will use them for rating weighting.")
    required_cols = ["user_idx", "anime_idx", "rating"] + [c for c in one_hot_status_cols if c in train_df.columns]
else:
    print("One-hot watching_status columns NOT found — falling back to rating-only weighting.")
    required_cols = ["user_idx", "anime_idx", "rating"]

for c in ["user_idx", "anime_idx", "rating"]:
    for df, name in [(train_df, "train"), (val_df, "val"), (test_df, "test")]:
        if c not in df.columns:
            raise ValueError(f"Missing column '{c}' in {name}.csv")

train_df = train_df[[c for c in required_cols if c in train_df.columns]].copy()
val_df   = val_df  [[c for c in required_cols if c in val_df.columns]].copy()
test_df  = test_df [[c for c in required_cols if c in test_df.columns]].copy()

for df in [train_df, val_df, test_df]:
    df["user_idx"]  = df["user_idx"].astype(int)
    df["anime_idx"] = df["anime_idx"].astype(int)
    df["rating"]    = df["rating"].astype(float)

NUM_USERS     = int(max(train_df.user_idx.max(), val_df.user_idx.max(), test_df.user_idx.max())) + 1
MAX_ANIME_IDX = int(max(train_df.anime_idx.max(), val_df.anime_idx.max(), test_df.anime_idx.max()))

print("Train:", train_df.shape, "Val:", val_df.shape, "Test:", test_df.shape)
print("NUM_USERS:", NUM_USERS) # user num
print("MAX_ANIME_IDX:", MAX_ANIME_IDX) # anime num

One-hot watching_status columns found — will use them for rating weighting.
Train: (82782709, 8) Val: (10347839, 8) Test: (10347839, 8)
NUM_USERS: 292565
MAX_ANIME_IDX: 13009


# 5.Ratings Weighted by Watching Status

In [5]:
# -------------------------------------------------------
# [watching_status weight map for one-hot encoded columns]
# -------------------------------------------------------
def get_status_weight_from_onehot_row(row):
    """Map a row with one-hot encoded watching_status to a weight in [0, 1]."""
    if "watch_status_Completed" in row and row["watch_status_Completed"] == 1.0:
        return 1.0   # Completed
    elif "watch_status_Currently Watching" in row and row["watch_status_Currently Watching"] == 1.0:
        return 0.8   # Watching
    elif "watch_status_On Hold" in row and row["watch_status_On Hold"] == 1.0:
        return 0.4   # On-hold
    elif "watch_status_Dropped" in row and row["watch_status_Dropped"] == 1.0:
        return 0.1   # Dropped
    elif "watch_status_Plan to Watch" in row and row["watch_status_Plan to Watch"] == 1.0:
        return 0.2   # Plan to watch
    else:
        return 0.5   # Default for unknown/other states

if has_status:
    train_df["status_weight"] = train_df.apply(get_status_weight_from_onehot_row, axis=1)
    # effective_rating = raw rating * status_weight
    # This makes a dropped-anime rating count much less toward positives
    train_df["effective_rating"] = train_df["rating"] * train_df["status_weight"]
else:
    train_df["effective_rating"] = train_df["rating"]

print("effective_rating stats:")
print(train_df["effective_rating"].describe())

effective_rating stats:
count    8.278271e+07
mean     4.305094e+00
std      3.886927e+00
min      0.000000e+00
25%      0.000000e+00
50%      6.000000e+00
75%      8.000000e+00
max      1.000000e+01
Name: effective_rating, dtype: float64


# 6. Anime Feature Matrix

In [6]:
# ---- Load & align anime feature matrix (unchanged from original model) ----
master_anime = np.load(MASTER_ANIME_PATH).astype(np.float32)
print("Original master_anime shape:", master_anime.shape)

if master_anime.shape[0] == MAX_ANIME_IDX:
    master_anime = np.vstack([
        np.zeros((1, master_anime.shape[1]), dtype=np.float32),
        master_anime
    ])
elif master_anime.shape[0] == MAX_ANIME_IDX + 1:
    pass
else:
    raise ValueError(
        f"MASTER_ANIME_TOWER_FEATURES.npy shape {master_anime.shape[0]} does not match "
        f"MAX_ANIME_IDX={MAX_ANIME_IDX}."
    )

NUM_ITEMS    = master_anime.shape[0]
ITEM_FEAT_DIM = master_anime.shape[1]

master_anime = np.nan_to_num(master_anime, nan=0.0, posinf=0.0, neginf=0.0)

if NUM_ITEMS > 1:
    item_mu    = master_anime[1:].mean(axis=0, keepdims=True)
    item_sigma = master_anime[1:].std(axis=0, keepdims=True) + 1e-6
    master_anime[1:] = (master_anime[1:] - item_mu) / item_sigma

print("Aligned master_anime shape:", master_anime.shape)

Original master_anime shape: (13010, 833)
Aligned master_anime shape: (13010, 833)


# 7.User Feature Matrix

In [7]:
# -------------------------------------------------------
# [Build user feature matrix using effective_rating]
# instead of raw rating for the preference vector weighting.
# POS_THRESHOLD check is also done on effective_rating so that
# a dropped anime rated 9 (effective ~0.9) no longer qualifies.
# -------------------------------------------------------
train_pos_df = train_df[train_df["effective_rating"] >= POS_THRESHOLD].copy()
if len(train_pos_df) == 0:
    raise ValueError(
        f"No positive samples found with effective_rating >= {POS_THRESHOLD}. "
        "Consider lowering POS_THRESHOLD or reviewing status weights."
    )

# basic user stats (still from raw rating for interpretability)
user_stats_df = train_df.groupby("user_idx").agg(
    user_rating_count=("rating", "count"),
    user_rating_mean=("rating", "mean"),
    user_rating_std=("rating", "std")
).fillna(0.0)

# preference vector: weight by effective_rating above threshold
item_pref_sum    = np.zeros((NUM_USERS, ITEM_FEAT_DIM), dtype=np.float32)
item_pref_weight = np.zeros(NUM_USERS, dtype=np.float32)

for row in train_pos_df.itertuples(index=False):
    u = int(row.user_idx)
    i = int(row.anime_idx)
    r = float(row.effective_rating)

    if i <= 0 or i >= NUM_ITEMS:
        continue

    # weight = excess above threshold (same as Caleb's formula but on effective_rating)
    w = max(r - POS_THRESHOLD + 1.0, 1.0)
    item_pref_sum[u]    += w * master_anime[i]
    item_pref_weight[u] += w

user_pref_vec = np.zeros((NUM_USERS, ITEM_FEAT_DIM), dtype=np.float32)
nonzero_mask  = item_pref_weight > 0
user_pref_vec[nonzero_mask] = (
    item_pref_sum[nonzero_mask] / item_pref_weight[nonzero_mask][:, None]
)

user_basic = np.zeros((NUM_USERS, user_stats_df.shape[1]), dtype=np.float32)
user_basic[user_stats_df.index.values] = user_stats_df.values.astype(np.float32)

user_dense = np.concatenate([user_basic, user_pref_vec], axis=1)
user_dense = np.nan_to_num(user_dense, nan=0.0, posinf=0.0, neginf=0.0)

nz = (np.abs(user_dense).sum(axis=1) > 0)
if nz.any():
    mu    = user_dense[nz].mean(axis=0, keepdims=True)
    sigma = user_dense[nz].std(axis=0, keepdims=True) + 1e-6
    user_dense[nz] = (user_dense[nz] - mu) / sigma

USER_FEAT_DIM = user_dense.shape[1]
print("user_basic shape:",    user_basic.shape)
print("user_pref_vec shape:", user_pref_vec.shape)
print("user_dense shape:",    user_dense.shape)
print("Train positives (effective_rating >= threshold):", len(train_pos_df))

user_basic shape: (292565, 3)
user_pref_vec shape: (292565, 833)
user_dense shape: (292565, 836)
Train positives (effective_rating >= threshold): 41704079


In [8]:
# ---- Seen maps and positive sets ----
user_seen_train = train_pos_df.groupby("user_idx")["anime_idx"].apply(set).to_dict()

val_pos_df      = val_df[val_df["rating"] >= POS_THRESHOLD].copy()
val_pos_by_user = val_pos_df.groupby("user_idx")["anime_idx"].apply(set).to_dict()

test_pos_df      = test_df[test_df["rating"] >= POS_THRESHOLD].copy()
test_pos_by_user = test_pos_df.groupby("user_idx")["anime_idx"].apply(set).to_dict()

print("Train positives:", train_pos_df.shape)
print("Val positives:",   val_pos_df.shape)
print("Test positives:",  test_pos_df.shape)


Train positives: (41704079, 10)
Val positives: (5446278, 8)
Test positives: (5444205, 8)


# 8. Dataset & DataLoader

In [9]:
# ---- Dataset & DataLoader (unchanged) ----
class TwoTowerBPRDataset(Dataset):
    def __init__(self, pos_df, user_seen_map, num_items):
        self.users     = pos_df["user_idx"].to_numpy(np.int64)
        self.pos_items = pos_df["anime_idx"].to_numpy(np.int64)
        self.user_seen = user_seen_map
        self.num_items = num_items

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        u = int(self.users[idx])
        p = int(self.pos_items[idx])
        seen = self.user_seen.get(u, set())
        n = np.random.randint(1, self.num_items)
        while n in seen:
            n = np.random.randint(1, self.num_items)
        return (
            torch.tensor(u, dtype=torch.long),
            torch.tensor(p, dtype=torch.long),
            torch.tensor(n, dtype=torch.long)
        )

train_loader = DataLoader(
    TwoTowerBPRDataset(train_pos_df, user_seen_train, NUM_ITEMS),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda")
)

#9. Sampled Ranking Evaluation Function

In [10]:
# ---- Sampled ranking evaluation (used during training loop only) ----
def _dcg_at_k(binary_hits):
    if len(binary_hits) == 0:
        return 0.0
    denom = np.log2(np.arange(2, len(binary_hits) + 2))
    return float((binary_hits / denom).sum())


@torch.no_grad()
def evaluate_ranking_at_k(
    model, pos_by_user, seen_items_by_user,
    k=10, max_users=1000, neg_samples=1000
):
    if len(pos_by_user) == 0:
        return 0.0, 0.0, 0.0, 0.0

    model.eval()
    users = list(pos_by_user.keys())
    if max_users is not None and len(users) > max_users:
        users = list(np.random.choice(users, size=max_users, replace=False))

    precisions, recalls, hitrates, ndcgs = [], [], [], []

    for u in users:
        u = int(u)
        pos_items = np.array(list(pos_by_user.get(u, set())), dtype=np.int64)
        if len(pos_items) == 0:
            continue

        seen = seen_items_by_user.get(u, set())
        neg  = np.random.randint(1, NUM_ITEMS, size=neg_samples, dtype=np.int64)
        neg  = np.array(
            [x for x in neg if x not in seen and x not in pos_by_user.get(u, set())],
            dtype=np.int64
        )
        if len(neg) == 0:
            continue

        candidates = np.unique(np.concatenate([pos_items, neg]))
        u_tensor   = torch.full((len(candidates),), u, dtype=torch.long, device=DEVICE)
        i_tensor   = torch.tensor(candidates, dtype=torch.long, device=DEVICE)
        scores     = model.score(u_tensor, i_tensor).detach().cpu().numpy()

        top_idx   = np.argsort(scores)[-k:][::-1]
        top_items = candidates[top_idx]

        hits     = np.isin(top_items, pos_items).astype(np.float32)
        num_hits = float(hits.sum())

        prec = num_hits / k
        rec  = num_hits / max(len(pos_items), 1)
        hr   = 1.0 if num_hits > 0 else 0.0
        dcg  = _dcg_at_k(hits)
        idcg = _dcg_at_k(np.ones(min(k, len(pos_items)), dtype=np.float32))
        ndcg = (dcg / idcg) if idcg > 0 else 0.0

        precisions.append(prec)
        recalls.append(rec)
        hitrates.append(hr)
        ndcgs.append(ndcg)

    if not precisions:
        return 0.0, 0.0, 0.0, 0.0

    return (
        float(np.mean(precisions)),
        float(np.mean(recalls)),
        float(np.mean(hitrates)),
        float(np.mean(ndcgs))
    )


#10. Model Definition

In [11]:
# ---- Model definition (unchanged) ----
class TwoTowerSimple(nn.Module):
    def __init__(self, num_users, num_items, user_dense_np, item_dense_np):
        super().__init__()

        self.user_id_emb = nn.Embedding(num_users, EMB_DIM)
        self.item_id_emb = nn.Embedding(num_items, EMB_DIM)

        self.register_buffer("user_dense_table", torch.tensor(user_dense_np, dtype=torch.float32))
        self.register_buffer("item_dense_table", torch.tensor(item_dense_np, dtype=torch.float32))

        user_in_dim = EMB_DIM + self.user_dense_table.shape[1]
        item_in_dim = EMB_DIM + self.item_dense_table.shape[1]

        self.user_mlp = nn.Sequential(
            nn.Linear(user_in_dim, HIDDEN_DIM),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_DIM, EMB_DIM)
        )
        self.item_mlp = nn.Sequential(
            nn.Linear(item_in_dim, HIDDEN_DIM),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_DIM, EMB_DIM)
        )

        self.user_bias   = nn.Embedding(num_users, 1)
        self.item_bias   = nn.Embedding(num_items, 1)
        self.global_bias = nn.Parameter(torch.zeros(1))

        nn.init.normal_(self.user_id_emb.weight, std=0.02)
        nn.init.normal_(self.item_id_emb.weight, std=0.02)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def user_vector(self, u_idx):
        u_input = torch.cat([self.user_id_emb(u_idx), self.user_dense_table[u_idx]], dim=1)
        return self.user_mlp(u_input)

    def item_vector(self, i_idx):
        i_input = torch.cat([self.item_id_emb(i_idx), self.item_dense_table[i_idx]], dim=1)
        return self.item_mlp(i_input)

    def score(self, u_idx, i_idx):
        u_vec = self.user_vector(u_idx)
        i_vec = self.item_vector(i_idx)
        dot   = (u_vec * i_vec).sum(dim=1)
        return dot + self.user_bias(u_idx).squeeze(-1) + self.item_bias(i_idx).squeeze(-1) + self.global_bias

# 11. Training Loop

In [13]:
# ---- Training loop ----
model = TwoTowerSimple(
    num_users=NUM_USERS,
    num_items=NUM_ITEMS,
    user_dense_np=user_dense,
    item_dense_np=master_anime
).to(DEVICE)

optimizer        = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
best_metric      = -np.inf
best_state       = None
best_epoch       = -1
patience_counter = 0

print(f"Start training | users={NUM_USERS} | items={NUM_ITEMS} | POS_THRESHOLD={POS_THRESHOLD}")

for epoch in range(1, EPOCHS + 1):
    model.train()
    losses = []

    for u, p, n in train_loader:
        u = u.to(DEVICE, non_blocking=True)
        p = p.to(DEVICE, non_blocking=True)
        n = n.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()
        pos_score = model.score(u, p)
        neg_score = model.score(u, n)
        loss = -F.logsigmoid(pos_score - neg_score).mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        losses.append(loss.item())

    val_p, val_r, val_hr, val_ndcg = evaluate_ranking_at_k(
        model=model,
        pos_by_user=val_pos_by_user,
        seen_items_by_user=user_seen_train,
        k=EVAL_K,
        max_users=EVAL_MAX_USERS,
        neg_samples=EVAL_NEG_SAMPLES
    )

    print(
        f"Epoch {epoch:02d} | bpr_loss={np.mean(losses):.4f} | "
        f"Val P@{EVAL_K}={val_p:.4f} | Val R@{EVAL_K}={val_r:.4f} | "
        f"Val HR@{EVAL_K}={val_hr:.4f} | Val NDCG@{EVAL_K}={val_ndcg:.4f}"
    )

    if val_ndcg > best_metric:
        best_metric      = val_ndcg
        best_epoch       = epoch
        best_state       = copy.deepcopy(model.state_dict())
        patience_counter = 0
        print(f"  New best model at epoch {epoch}")
    else:
        patience_counter += 1
        print(f"  No improvement. Patience: {patience_counter}/{PATIENCE}")

    if patience_counter >= PATIENCE:
        print("Early stopping triggered.")
        break

if best_state is not None:
    model.load_state_dict(best_state)
    print(f"\nLoaded best model from epoch {best_epoch} with Val NDCG@{EVAL_K}={best_metric:.4f}")


Start training | users=292565 | items=13010 | POS_THRESHOLD=6.0
Epoch 01 | bpr_loss=0.0977 | Val P@10=0.4959 | Val R@10=0.4314 | Val HR@10=0.9740 | Val NDCG@10=0.6281
  New best model at epoch 1
Epoch 02 | bpr_loss=0.0871 | Val P@10=0.5078 | Val R@10=0.4173 | Val HR@10=0.9680 | Val NDCG@10=0.6302
  New best model at epoch 2
Epoch 03 | bpr_loss=0.0845 | Val P@10=0.5013 | Val R@10=0.4305 | Val HR@10=0.9760 | Val NDCG@10=0.6300
  No improvement. Patience: 1/2
Epoch 04 | bpr_loss=0.0831 | Val P@10=0.5090 | Val R@10=0.4411 | Val HR@10=0.9730 | Val NDCG@10=0.6407
  New best model at epoch 4
Epoch 05 | bpr_loss=0.0823 | Val P@10=0.5367 | Val R@10=0.4124 | Val HR@10=0.9670 | Val NDCG@10=0.6513
  New best model at epoch 5

Loaded best model from epoch 5 with Val NDCG@10=0.6513


# 12. Sampled Test Evaluation

In [14]:
# ---- Sampled test evaluation (overall only — full catalog + segments done in next cell) ----
test_p, test_r, test_hr, test_ndcg = evaluate_ranking_at_k(
    model=model,
    pos_by_user=test_pos_by_user,
    seen_items_by_user=user_seen_train,
    k=EVAL_K,
    max_users=EVAL_MAX_USERS,
    neg_samples=EVAL_NEG_SAMPLES
)

print("\n===== Final Test Metrics — Sampled (overall) =====")
print(f"P@{EVAL_K}    = {test_p:.4f}")
print(f"R@{EVAL_K}    = {test_r:.4f}")
print(f"HR@{EVAL_K}   = {test_hr:.4f}")
print(f"NDCG@{EVAL_K} = {test_ndcg:.4f}")
print("(sampled against 1000 negatives — inflated vs full catalog)")



===== Final Test Metrics — Sampled (overall) =====
P@10    = 0.5249
R@10    = 0.4292
HR@10   = 0.9830
NDCG@10 = 0.6568
(sampled against 1000 negatives — inflated vs full catalog)


#13. Full Catalog Evaluation

In [15]:
# ============================================================
# Full Catalog Evaluation + Analysis (same as Caleb's version for comparison)
#   - 4-way user activity segments (light/medium/heavy/power) via qcut
#   - Hit rate by ground-truth popularity bucket (head_mid vs long_tail)
#   - Recommendation bias / coverage summary
# ============================================================
from collections import Counter

ANALYSIS_K                   = EVAL_K
ANALYSIS_MAX_USERS           = 300
PERSONALIZATION_SAMPLE_USERS = 200
LONG_TAIL_QUANTILE           = 0.80

# ------------------------------------------------------------
# A. Get full-catalog top-K recommendations for selected users
# ------------------------------------------------------------
@torch.no_grad()
def get_topk_recommendations_full_catalog(model, user_ids, seen_items_by_user, k=10):
    model.eval()
    item_ids_np = np.arange(1, NUM_ITEMS, dtype=np.int64)
    all_item_vecs = []
    for start in range(0, len(item_ids_np), 2048):
        batch_ids = torch.tensor(
            item_ids_np[start:start + 2048], dtype=torch.long, device=DEVICE
        )
        all_item_vecs.append(model.item_vector(batch_ids))
    all_item_vecs = torch.cat(all_item_vecs, dim=0)

    results = {}
    for start in range(0, len(user_ids), 256):
        batch_users = user_ids[start:start + 256]
        u_tensor  = torch.tensor(batch_users, dtype=torch.long, device=DEVICE)
        user_vecs = model.user_vector(u_tensor)
        scores    = torch.matmul(user_vecs, all_item_vecs.T).cpu().numpy()
        for row_idx, u in enumerate(batch_users):
            u = int(u)
            seen = seen_items_by_user.get(u, set())
            row_scores = scores[row_idx].copy()
            if len(seen) > 0:
                row_scores[np.isin(item_ids_np, list(seen))] = -1e12
            top_idx = np.argpartition(row_scores, -k)[-k:]
            top_idx = top_idx[np.argsort(row_scores[top_idx])[::-1]]
            results[u] = item_ids_np[top_idx].tolist()
    return results


# ------------------------------------------------------------
# B. Compute user-level ranking metrics from recommendation dict
# ------------------------------------------------------------
def user_level_metrics_from_recs(recs_by_user, gt_by_user, k=10):
    rows = []
    for u, recs in recs_by_user.items():
        gt = gt_by_user.get(u, set())
        if len(gt) == 0:
            continue
        topk     = recs[:k]
        hits     = np.isin(topk, list(gt)).astype(np.float32)
        num_hits = float(hits.sum())
        precision = num_hits / k
        recall    = num_hits / len(gt)
        hitrate   = 1.0 if num_hits > 0 else 0.0
        denom     = np.log2(np.arange(2, len(topk) + 2))
        dcg       = float((hits / denom).sum())
        ideal     = np.ones(min(k, len(gt)), dtype=np.float32)
        idcg      = float((ideal / np.log2(np.arange(2, len(ideal) + 2))).sum())
        ndcg      = dcg / idcg if idcg > 0 else 0.0
        rows.append({
            "user_idx": u, "precision": precision, "recall": recall,
            "hitrate": hitrate, "ndcg": ndcg, "num_test_positives": len(gt)
        })
    return pd.DataFrame(rows)


# ------------------------------------------------------------
# C. Select users and get recommendations
# ------------------------------------------------------------
analysis_users = list(test_pos_by_user.keys())
if ANALYSIS_MAX_USERS is not None and len(analysis_users) > ANALYSIS_MAX_USERS:
    np.random.seed(SEED)
    analysis_users = list(np.random.choice(analysis_users, size=ANALYSIS_MAX_USERS, replace=False))

print("Running recommendation audit on", len(analysis_users), "users...")
audit_recs = get_topk_recommendations_full_catalog(
    model=model, user_ids=analysis_users,
    seen_items_by_user=user_seen_train, k=ANALYSIS_K
)
audit_metrics_df = user_level_metrics_from_recs(
    recs_by_user=audit_recs, gt_by_user=test_pos_by_user, k=ANALYSIS_K
)
print("\n=== User-Level Audit Metrics ===")
print(audit_metrics_df[["precision", "recall", "hitrate", "ndcg"]].mean().round(4))


# ------------------------------------------------------------
# D. User activity segment analysis (4-way qcut — same as Caleb)
# ------------------------------------------------------------
train_user_activity = train_df.groupby("user_idx").size().rename("train_interactions")
train_user_activity = train_user_activity.reindex(range(NUM_USERS), fill_value=0)

activity_df = pd.DataFrame({
    "user_idx":           analysis_users,
    "train_interactions": [train_user_activity.get(u, 0) for u in analysis_users]
})
activity_df["activity_segment"] = pd.qcut(
    activity_df["train_interactions"].rank(method="first"),
    q=4, labels=["light", "medium", "heavy", "power"], duplicates="drop"
)
audit_metrics_df = audit_metrics_df.merge(activity_df, on="user_idx", how="left")

segment_perf   = audit_metrics_df.groupby("activity_segment", observed=False)[
    ["precision", "recall", "hitrate", "ndcg"]
].mean()
segment_counts = audit_metrics_df.groupby("activity_segment", observed=False).size().rename("num_users")
print("\n=== Performance by User Activity Segment ===")
print(pd.concat([segment_perf, segment_counts], axis=1).round(4))


# ------------------------------------------------------------
# E. Popularity statistics from TRAIN
# ------------------------------------------------------------
item_pop         = train_df["anime_idx"].value_counts().rename("train_popularity")
item_pop         = item_pop.reindex(range(NUM_ITEMS), fill_value=0)
nonzero_item_pop = item_pop[item_pop.index != 0]
pop_threshold    = nonzero_item_pop.quantile(LONG_TAIL_QUANTILE)
item_bucket      = pd.Series(index=nonzero_item_pop.index, dtype="object")
item_bucket[nonzero_item_pop >= pop_threshold] = "head_mid"
item_bucket[nonzero_item_pop <  pop_threshold] = "long_tail"
pop_prob         = nonzero_item_pop / nonzero_item_pop.sum()
item_novelty     = -np.log(pop_prob + 1e-12)


# ------------------------------------------------------------
# F. Hit-rate by ground-truth popularity bucket
# ------------------------------------------------------------
bucket_hit_rows = []
for u, recs in audit_recs.items():
    gt   = test_pos_by_user.get(u, set())
    topk = set(recs[:ANALYSIS_K])
    for item in gt:
        bucket_hit_rows.append({
            "user_idx": u, "anime_idx": item,
            "popularity_bucket": item_bucket.get(item, "head_mid"),
            "hit_at_k": 1 if item in topk else 0
        })
bucket_hit_df = pd.DataFrame(bucket_hit_rows)
print("\n=== Hit Rate by Ground-Truth Popularity Bucket ===")
if len(bucket_hit_df) > 0:
    pop_bucket_perf   = bucket_hit_df.groupby("popularity_bucket")["hit_at_k"].mean()
    pop_bucket_counts = bucket_hit_df.groupby("popularity_bucket").size().rename("num_items")
    print(pd.concat([pop_bucket_perf.rename("hit_rate"), pop_bucket_counts], axis=1).round(4))


# ------------------------------------------------------------
# G. Recommendation distribution / bias metrics
# ------------------------------------------------------------
all_recommended_items = [i for u, recs in audit_recs.items() for i in recs[:ANALYSIS_K]]
rec_counter        = Counter(all_recommended_items)
catalog_coverage   = len(rec_counter) / (NUM_ITEMS - 1)
rec_freq           = np.array(list(rec_counter.values()), dtype=np.float64)
rec_share          = rec_freq / rec_freq.sum()
recommendation_hhi = float(np.sum(rec_share ** 2))
avg_rec_popularity = float(np.mean([item_pop.get(i, 0) for i in all_recommended_items]))
rec_long_tail_share = float(np.mean([
    1.0 if item_bucket.get(i, "head_mid") == "long_tail" else 0.0
    for i in all_recommended_items
]))
avg_rec_novelty = float(np.mean([item_novelty.get(i, 0.0) for i in all_recommended_items]))

all_gt_items       = [i for u in analysis_users for i in test_pos_by_user.get(u, set())]
avg_gt_popularity  = float(np.mean([item_pop.get(i, 0) for i in all_gt_items])) if all_gt_items else 0.0
gt_long_tail_share = float(np.mean([
    1.0 if item_bucket.get(i, "head_mid") == "long_tail" else 0.0
    for i in all_gt_items
])) if all_gt_items else 0.0


# ------------------------------------------------------------
# H. Personalization
# ------------------------------------------------------------
sample_users = list(audit_recs.keys())
if len(sample_users) > PERSONALIZATION_SAMPLE_USERS:
    np.random.seed(SEED)
    sample_users = list(np.random.choice(sample_users, size=PERSONALIZATION_SAMPLE_USERS, replace=False))
jaccards = []
for i in range(len(sample_users)):
    set_i = set(audit_recs[sample_users[i]][:ANALYSIS_K])
    for j in range(i + 1, len(sample_users)):
        set_j = set(audit_recs[sample_users[j]][:ANALYSIS_K])
        union = len(set_i | set_j)
        if union > 0:
            jaccards.append(len(set_i & set_j) / union)
personalization = 1.0 - (float(np.mean(jaccards)) if jaccards else 0.0)


# ------------------------------------------------------------
# I. Summary table
# ------------------------------------------------------------
summary_df = pd.DataFrame({
    "metric": [
        "catalog_coverage_at_k", "recommendation_hhi",
        "avg_recommended_item_popularity", "avg_ground_truth_item_popularity",
        "popularity_gap_rec_minus_truth", "recommended_long_tail_share",
        "ground_truth_long_tail_share", "long_tail_gap_rec_minus_truth",
        "avg_recommendation_novelty", "personalization_1_minus_avg_jaccard"
    ],
    "value": [
        catalog_coverage, recommendation_hhi,
        avg_rec_popularity, avg_gt_popularity,
        avg_rec_popularity - avg_gt_popularity, rec_long_tail_share,
        gt_long_tail_share, rec_long_tail_share - gt_long_tail_share,
        avg_rec_novelty, personalization
    ]
})
print("\n=== Recommendation Bias / Coverage Summary ===")
print(summary_df.round(4))


Running recommendation audit on 300 users...

=== User-Level Audit Metrics ===
precision    0.1753
recall       0.1437
hitrate      0.7400
ndcg         0.2126
dtype: float64

=== Performance by User Activity Segment ===
                  precision  recall  hitrate    ndcg  num_users
activity_segment                                               
light                0.1053  0.2232   0.6133  0.1845         75
medium               0.1333  0.1436   0.6800  0.1666         75
heavy                0.2080  0.1358   0.8400  0.2309         75
power                0.2547  0.0722   0.8267  0.2687         75

=== Hit Rate by Ground-Truth Popularity Bucket ===
                   hit_rate  num_items
popularity_bucket                     
head_mid             0.1030       4988
long_tail            0.0142        847

=== Recommendation Bias / Coverage Summary ===
                                metric       value
0                catalog_coverage_at_k      0.0791
1                   recommendation_hhi